## Basic imports and variable creations

In [ ]:
# Cell 2
import os, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import platform
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
except Exception:
    SentenceTransformer = None
    HAS_SBERT = False

try:
    import hnswlib
    HAS_HNSW = True
except Exception:
    hnswlib = None
    HAS_HNSW = False

print("sklearn", sklearn.__version__, "numpy", np.__version__, "pandas", pd.__version__,
      "sentence-transformers:", HAS_SBERT, "hnswlib:", HAS_HNSW)


sklearn 1.7.2 numpy 2.2.6 pandas 2.3.2 sentence-transformers: False hnswlib: True


### Define location override rule to make sure the model understands the online businesses and other locations

In [21]:
# --- Location override rule: Online > explicit district > model_loc ---
import re

DISTRICTS = [
    "Colombo","Gampaha","Kalutara","Kandy","Matale","Nuwara Eliya","Galle","Matara","Hambantota",
    "Jaffna","Kilinochchi","Mannar","Vavuniya","Mullaitivu","Batticaloa","Ampara","Trincomalee",
    "Kurunegala","Puttalam","Anuradhapura","Polonnaruwa","Badulla","Monaragala","Ratnapura","Kegalle"
]
_DISTRICT_PATTERNS = {d: re.compile(rf"\b{re.escape(d.lower())}\b", re.IGNORECASE) for d in DISTRICTS}
_DISTRICT_CANON    = {d.lower(): d for d in DISTRICTS}

_ONLINE_PATTERNS = [
    r"online", r"e-?commerce", r"\bweb(app)?\b", r"website", r"internet", r"virtual", r"remote",
    r"cloud", r"\bsaas\b", r"software as a service", r"\bplatform\b", r"\bsoftware\b",
    r"\blms\b", r"\bapp\b", r"\bapi\b", r"marketplace", r"digital", r"subscription", r"online-only"
]
_ONLINE_RE = re.compile("|".join(_ONLINE_PATTERNS), re.IGNORECASE)

def infer_location_simple(text: str, model_loc: str | None = None) -> str | None:
    """
    Priority:
      1) explicit online/software signals -> 'Online'
      2) explicit Sri Lanka district in text -> that district
      3) otherwise keep model_loc
    """
    s = (text or "")
    if _ONLINE_RE.search(s):
        return "Online"
    s_low = s.lower()
    for d, pat in _DISTRICT_PATTERNS.items():
        if pat.search(s_low):
            return _DISTRICT_CANON[d.lower()]
    return model_loc


### call the dataset as df

In [22]:
df = pd.read_csv("sl_startup.csv")

### Create combined word + char TF-IDF featurizer

In [ ]:


word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1,2),
    max_features=30_000,
    lowercase=True,
    stop_words="english",
    strip_accents="unicode"
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,5),
    max_features=10_000,
    lowercase=True
)

featurizer = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

# Optionally reduce dimensionality for classifier speed
svd = TruncatedSVD(n_components=512, random_state=42)   # adjust down if memory constrained

featurizer_pipeline = Pipeline([
    ("tfidf", featurizer),
    ("svd", svd)
])

# Fit featurizer on all descriptions
texts = df["Description"].fillna("").astype(str).tolist()
print("Fitting featurizer on", len(texts), "texts ...")
start = time.time()
featurizer_pipeline.fit(texts)
print("Featurizer fit done (sec):", time.time() - start)


Fitting featurizer on 10000 texts ...
Featurizer fit done (sec): 16.36228919029236


### Prepare Lables for training testing

In [ ]:

y_cat = df["Category"].astype(str).fillna("").values
y_loc = df["Location"].astype(str).fillna("").values

enc_cat = LabelEncoder(); y_cat_enc = enc_cat.fit_transform(y_cat)
enc_loc = LabelEncoder(); y_loc_enc = enc_loc.fit_transform(y_loc)

# Train/test split for evaluation
X_train, X_test, yc_train, yc_test, yl_train, yl_test = train_test_split(
    df["Description"].astype(str).tolist(), y_cat_enc, y_loc_enc, test_size=0.12, random_state=42, stratify=y_cat_enc
)

# Transform training features (fit already done on whole dataset above — or you can fit on train only)
Xv_train = featurizer_pipeline.transform(X_train)
Xv_test = featurizer_pipeline.transform(X_test)

# Classifiers
cat_clf = LogisticRegression(max_iter=200, solver="saga", n_jobs=-1, random_state=42)
loc_clf = LogisticRegression(max_iter=200, solver="saga", n_jobs=-1, random_state=42)

print("Training category head ...")
cat_clf.fit(Xv_train, yc_train)
print("Training location head ...")
loc_clf.fit(Xv_train, yl_train)

# Evaluate
yc_pred = cat_clf.predict(Xv_test)
yl_pred = loc_clf.predict(Xv_test)
print("Category accuracy:", accuracy_score(yc_test, yc_pred))
print("Location accuracy:", accuracy_score(yl_test, yl_pred))
print("Category classification report:")
print(classification_report(yc_test, yc_pred, target_names=enc_cat.inverse_transform(np.unique(yc_test))))


Training category head ...
Training location head ...
Category accuracy: 1.0
Location accuracy: 1.0
Category classification report:
                                      precision    recall  f1-score   support

                         Agriculture       1.00      1.00      1.00         8
                Arts & Entertainment       1.00      1.00      1.00         8
              Beauty & Personal Care       1.00      1.00      1.00         7
                           Education       1.00      1.00      1.00         8
                Energy & Environment       1.00      1.00      1.00         7
                              Events       1.00      1.00      1.00         7
                 Finance & Insurance       1.00      1.00      1.00         7
                     Food & Beverage       1.00      1.00      1.00         6
                   Health & Wellness       1.00      1.00      1.00         7
                           Logistics       1.00      1.00      1.00         8
         

In [26]:
# Cell 6
EMBEDDING_BACKEND = None
embedding_vectors = None

if HAS_SBERT:
    # Use a small sbert model for speed. You can change model_name to a different HF model.
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    print("Computing SBERT embeddings using", model_name)
    sbert = SentenceTransformer(model_name)
    # compute for all corpus (text order must match corpus file saved later)
    all_texts = df["Description"].fillna("").astype(str).tolist()
    emb = sbert.encode(all_texts, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
    # optionally reduce dimension to speed ANN and save embedding_dim
    from sklearn.decomposition import TruncatedSVD as _SVD
    REDUCE_DIM = 256
    if emb.shape[1] > REDUCE_DIM:
        red = _SVD(n_components=REDUCE_DIM, random_state=42)
        emb_reduced = red.fit_transform(emb)
        embedding_vectors = emb_reduced.astype("float32")
    else:
        embedding_vectors = emb.astype("float32")
    EMBEDDING_BACKEND = {"type":"sbert", "model_name": model_name, "normalize": True}
else:
    # fallback: use featurizer pipeline outputs as embeddings (already SVD reduced to 512)
    print("SentenceTransformer not available — using featurizer SVD outputs as embedding vectors")
    all_texts = df["Description"].fillna("").astype(str).tolist()
    emb = featurizer_pipeline.transform(all_texts)  # dense array
    # ensure float32
    embedding_vectors = np.asarray(emb, dtype="float32")
    EMBEDDING_BACKEND = {"type":"tfidf_svd", "info": "use featurizer pipeline from featurizer_cls.joblib"}
print("Embedding vectors shape:", embedding_vectors.shape)


SentenceTransformer not available — using featurizer SVD outputs as embedding vectors
Embedding vectors shape: (10000, 512)


## Building ANN path and saving HNSW

In [31]:
# Cell 7 (fixed)
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
import numpy as np, joblib, time

OUT_DIR = Path("artifacts_n")              # <-- make it a Path, not a str
OUT_DIR.mkdir(parents=True, exist_ok=True)

ANN_PATH = OUT_DIR / "neighbors.index"
ANN_JOBLIB_PATH = OUT_DIR / "neighbors.joblib"

if HAS_HNSW:
    print("Building HNSW index (hnswlib)...")
    dim = int(embedding_vectors.shape[1])
    p = hnswlib.Index(space='cosine', dim=dim)
    p.init_index(max_elements=int(embedding_vectors.shape[0]), ef_construction=200, M=64)
    p.add_items(embedding_vectors.astype('float32'), np.arange(embedding_vectors.shape[0]))
    p.set_ef(50)
    p.save_index(str(ANN_PATH))
    print("Saved HNSW index to", ANN_PATH)
    ann_backend = {"type": "hnswlib", "path": str(ANN_PATH)}
else:
    print("Building sklearn NearestNeighbors fallback index (joblib)...")
    # If your sklearn warns about n_jobs, remove it:
    try:
        knn = NearestNeighbors(n_neighbors=30, algorithm="auto", metric="cosine", n_jobs=-1)
    except TypeError:
        knn = NearestNeighbors(n_neighbors=30, algorithm="auto", metric="cosine")
    knn.fit(embedding_vectors.astype('float32'))
    joblib.dump(knn, ANN_JOBLIB_PATH)
    print("Saved NearestNeighbors to", ANN_JOBLIB_PATH)
    ann_backend = {"type": "sklearn_knn", "path": str(ANN_JOBLIB_PATH)}


Building HNSW index (hnswlib)...
Saved HNSW index to artifacts_n\neighbors.index


### Create variables to save the model as segmented files

In [32]:
# Cell 8
# File map (names backend expects)
FILES = {
    "featurizer_cls": OUT_DIR / "featurizer_cls.joblib",
    "cat_head": OUT_DIR / "cat_head.joblib",
    "loc_head": OUT_DIR / "loc_head.joblib",
    "encoders": OUT_DIR / "label_encoders.joblib",
    "embedder_json": OUT_DIR / "embedder.json",
    "ann_joblib": OUT_DIR / "neighbors.joblib",
    "ann_index": OUT_DIR / "neighbors.index",
    "corpus_parquet": OUT_DIR / "corpus.parquet",
    "corpus_csv": OUT_DIR / "corpus.csv",
    "meta": OUT_DIR / "meta.json"
}

print("Saving featurizer pipeline to", FILES["featurizer_cls"])
joblib.dump(featurizer_pipeline, FILES["featurizer_cls"])

print("Saving heads ...")
joblib.dump(cat_clf, FILES["cat_head"])
joblib.dump(loc_clf, FILES["loc_head"])

print("Saving label encoders as dict {'cat': enc_cat, 'loc': enc_loc}")
joblib.dump({"cat": enc_cat, "loc": enc_loc}, FILES["encoders"])

# Save embedder.json (do not store SBERT model object)
embed_cfg = {
    "backend": EMBEDDING_BACKEND,
    "ann": ann_backend,
    "embedder_dim": int(embedding_vectors.shape[1]),
}
with open(FILES["embedder_json"], "w", encoding="utf8") as f:
    json.dump(embed_cfg, f, indent=2)
print("Saved embedder.json ->", FILES["embedder_json"])

# Save ANN whichever path: if hnsw created earlier, file is already stored. If sklearn, we saved neighbor joblib.
if not HAS_HNSW and ANN_JOBLIB_PATH.exists():
    # neighbor saved already
    pass

# Save corpus: try parquet, fallback csv
try:
    df.to_parquet(FILES["corpus_parquet"], index=False)
    corpus_path = str(FILES["corpus_parquet"])
    print("Saved corpus.parquet ->", corpus_path)
except Exception as e:
    df.to_csv(FILES["corpus_csv"], index=False)
    corpus_path = str(FILES["corpus_csv"])
    print("Parquet failed; saved CSV ->", corpus_path)

# Save meta info
meta = {
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "rows": int(len(df)),
    "featurizer": "tfidf(word ngram 1-2) + tfidf(char_wb 3-5) + TruncatedSVD(512)",
    "embedding_backend": EMBEDDING_BACKEND,
    "ann_backend": ann_backend,
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
with open(FILES["meta"], "w", encoding="utf8") as f:
    json.dump(meta, f, indent=2)
print("Saved meta ->", FILES["meta"])
print("\nArtifacts saved under:", OUT_DIR)


Saving featurizer pipeline to artifacts_n\featurizer_cls.joblib
Saving heads ...
Saving label encoders as dict {'cat': enc_cat, 'loc': enc_loc}
Saved embedder.json -> artifacts_n\embedder.json
Saved corpus.parquet -> artifacts_n\corpus.parquet
Saved meta -> artifacts_n\meta.json

Artifacts saved under: artifacts_n


### Load saved model files

In [30]:
# Cell 9
print("Loading saved featurizer and heads for a quick sanity check...")
fe = joblib.load(FILES["featurizer_cls"])
cat_h = joblib.load(FILES["cat_head"])
loc_h = joblib.load(FILES["loc_head"])
encs = joblib.load(FILES["encoders"])   # dict with 'cat','loc'

# helper to run predict
def predict_text(text):
    xv = fe.transform([text])
    yc = cat_h.predict(xv)[0]
    yl = loc_h.predict(xv)[0]
    loc_final = infer_location_simple(text, model_loc=str(yl))
    print({"Category": yc, "Location": loc_final})
    cat = encs["cat"].inverse_transform([yc])[0]
    loc = encs["loc"].inverse_transform([yl])[0]
    
    return {"Category": str(cat), "Location": str(loc)}

samples = [
    "I want to start a clothing store for young adults in Galle",
    "Build a cloud LMS app for K-12 schools"
]
for s in samples:
    print("\nQ:", s)
    print("Pred:", predict_text(s))

# Neighbor check using ANN index / sklearn fallback
print("\nNeighbor query test (first sample) ...")
try:
    if HAS_HNSW and (OUT_DIR / "neighbors.index").exists():
        # load hnsw and query
        idx = hnswlib.Index(space='cosine', dim=embedding_vectors.shape[1])
        idx.load_index(str(OUT_DIR / "neighbors.index"))
        # use same embedding approach as training to embed a sample
        if HAS_SBERT:
            qvec = sbert.encode([samples[0]], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        else:
            qvec = fe.transform([samples[0]]).astype("float32")
        labels, dists = idx.knn_query(qvec, k=5)
        print("Neighbor IDs:", labels[0].tolist(), "dists:", dists[0].tolist())
    else:
        kn = joblib.load(OUT_DIR / "neighbors.joblib")
        if HAS_SBERT:
            qvec = sbert.encode([samples[0]], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        else:
            qvec = fe.transform([samples[0]]).astype("float32")
        dists, idxs = kn.kneighbors(qvec, n_neighbors=5, return_distance=True)
        print("Neighbor idxs:", idxs[0].tolist(), "dists:", dists[0].tolist())
except Exception as e:
    print("Neighbor test failed:", e)


Loading saved featurizer and heads for a quick sanity check...

Q: I want to start a clothing store for young adults in Galle
{'Category': np.int64(153), 'Location': 'Galle'}
Pred: {'Category': 'warehousing', 'Location': 'Galle'}

Q: Build a cloud LMS app for K-12 schools
{'Category': np.int64(51), 'Location': 'Online'}
Pred: {'Category': 'education management', 'Location': 'Galle'}

Neighbor query test (first sample) ...
Neighbor IDs: [1110, 3725, 4469, 9386, 4245] dists: [0.6952825784683228, 0.7049623131752014, 0.705290675163269, 0.7072314620018005, 0.7077751159667969]


### sets where your saved model pieces live and whether to run locally or via a server, brings in the needed libraries, defines a tiny text cleaner so old pickled pipelines load safely, maps the expected artifact file paths

In [34]:
# ==== CONFIG ====
ART_DIR = r"artifacts_n"   # <- point to your artifacts folder
BACKEND_URL = None                       # e.g. "http://127.0.0.1:8000/api/ml/classify/" or leave None to use local artifacts

# ==== IMPORTS ====
import os, json, re, time, sys, types, joblib, cloudpickle as cp
import numpy as np
import pandas as pd
import requests

# ---- Shim so featurizer unpickles if it references __main__.TextNormalizer
class TextNormalizer:
    def __init__(self, lowercase=True, strip=True): self.lowercase, self.strip = lowercase, strip
    def fit(self, X, y=None): return self
    def transform(self, X):
        s = pd.Series(X, dtype=object).astype(str)
        if self.lowercase: s = s.str.lower()
        if self.strip: s = s.str.strip()
        return s
    def get_params(self, deep=True): return {"lowercase": self.lowercase, "strip": self.strip}
    def set_params(self, **p): [setattr(self, k, v) for k,v in p.items()]; return self

if "__main__" not in sys.modules:
    sys.modules["__main__"] = types.ModuleType("__main__")
setattr(sys.modules["__main__"], "TextNormalizer", TextNormalizer)

# ---- artifact paths
FILES = {
    "featurizer_cls_joblib": os.path.join(ART_DIR, "featurizer_cls.joblib"),
    "featurizer_cls_cpkl":   os.path.join(ART_DIR, "featurizer_cls.cpkl"),
    "cat_head":              os.path.join(ART_DIR, "cat_head.joblib"),
    "loc_head":              os.path.join(ART_DIR, "loc_head.joblib"),
    "encoders":              os.path.join(ART_DIR, "label_encoders.joblib"),
}

# ---- load artifacts (local path mode)
def _load_local():
    # featurizer pipeline
    try:
        feat = joblib.load(FILES["featurizer_cls_joblib"])
    except Exception:
        with open(FILES["featurizer_cls_cpkl"], "rb") as f:
            feat = cp.load(f)

    # heads
    cat_head = joblib.load(FILES["cat_head"])
    loc_head = joblib.load(FILES["loc_head"])

    # encoders (robust)
    enc_raw = joblib.load(FILES["encoders"])
    def extract_encoders(enc_obj):
        if isinstance(enc_obj, dict):
            lower = {str(k).lower(): v for k,v in enc_obj.items()}
            cat = lower.get("cat") or lower.get("category")
            loc = lower.get("loc") or lower.get("location")
            if cat is not None and loc is not None: return cat, loc
        if isinstance(enc_obj, (list, tuple)) and len(enc_obj) >= 2:
            return enc_obj[0], enc_obj[1]
        if hasattr(enc_obj, "encoders") and isinstance(enc_obj.encoders, dict):
            lower = {str(k).lower(): v for k,v in enc_obj.encoders.items()}
            cat = lower.get("cat") or lower.get("category")
            loc = lower.get("loc") or lower.get("location")
            if cat is not None and loc is not None: return cat, loc
        raise KeyError("Could not find label encoders for Category/Location in encoders artifact")
    enc_cat, enc_loc = extract_encoders(enc_raw)

    return feat, cat_head, loc_head, enc_cat, enc_loc

feat = cat_head = loc_head = enc_cat = enc_loc = None
if BACKEND_URL is None:
    feat, cat_head, loc_head, enc_cat, enc_loc = _load_local()

# ---- get all category names
if BACKEND_URL is None:
    all_cat_ids = np.arange(len(enc_cat.classes_))
    ALL_CATS = [str(x) for x in enc_cat.classes_]
else:
    # pull categories from backend by probing encoder via /ml/info (optional)
    try:
        info = requests.get(BACKEND_URL.replace("/classify/","/info/"), timeout=10).json()
        ALL_CATS = sorted(set(
            info.get("details",{}).get("meta",{}).get("all_categories", [])
        ))
        if not ALL_CATS:
            # fallback: send a hint list you keep yourself
            raise RuntimeError("No categories in meta; please provide manually.")
    except Exception:
        raise RuntimeError("Set BACKEND_URL to use server mode, or leave None and run locally with artifacts.")


### Generates two test prompts per category (one “Galle” physical, one online), runs the model (local or API), checks if cat/location are correct,

In [35]:
# Prompts per category
def mk_prompts(cat_name: str):
    # physical version (forces a district)
    p1 = f"I want to start a {cat_name} business in Galle."
    # online version (forces online signals)
    p2 = f"Build an online SaaS platform for {cat_name}."
    return p1, p2

def backend_predict(text):
    r = requests.post(BACKEND_URL, json={"text": text}, timeout=20)
    r.raise_for_status()
    pred = r.json()["predictions"][0]
    return str(pred.get("Category")), str(pred.get("Location"))

def local_predict(texts):
    X = pd.Series(texts, dtype=object)
    Xv = feat.transform(X)
    ycat = cat_head.predict(Xv)
    yloc = loc_head.predict(Xv)
    cats = enc_cat.inverse_transform(ycat)
    locs = enc_loc.inverse_transform(yloc)
    # apply same location override used in backend (light version)
    ONLINE_RE = re.compile(r"online|e-?commerce|\bsaas\b|cloud|virtual|remote|\bplatform\b|\blms\b|\bapp\b|marketplace|subscription", re.I)
    DISTRICTS = ["Colombo","Gampaha","Kalutara","Kandy","Matale","Nuwara Eliya","Galle","Matara","Hambantota","Jaffna","Kilinochchi","Mannar","Vavuniya","Mullaitivu","Batticaloa","Ampara","Trincomalee","Kurunegala","Puttalam","Anuradhapura","Polonnaruwa","Badulla","Monaragala","Ratnapura","Kegalle"]
    DIST_RE = {d: re.compile(rf"\b{re.escape(d)}\b", re.I) for d in DISTRICTS}

    outs = []
    for t, c, l in zip(texts, cats, locs):
        loc = str(l)
        if ONLINE_RE.search(t or ""):
            loc = "Online"
        else:
            for d, pat in DIST_RE.items():
                if pat.search(t or ""):
                    loc = d; break
        outs.append((str(c), loc))
    return outs

rows = []
start = time.time()

BATCH = 64
if BACKEND_URL is None:
    # local fast path; do batched featurization
    prompts = []
    owners  = []
    for cat in ALL_CATS:
        p1, p2 = mk_prompts(cat)
        prompts.extend([p1, p2])
        owners.extend([(cat, "physical"), (cat, "online")])
    preds = local_predict(prompts)
    for (cat, kind), (pc, pl), prompt in zip(owners, preds, prompts):
        rows.append({
            "TargetCategory": cat,
            "PromptType": kind,
            "Prompt": prompt,
            "PredCategory": pc,
            "PredLocation": pl,
            "CatMatch": (pc.lower() == cat.lower()),
            "LocIsOnline": (pl == "Online"),
            "LocIsGalle":  (pl == "Galle"),
        })
else:
    # backend mode; call API one by one (keeps it simple/reliable)
    for i, cat in enumerate(ALL_CATS, 1):
        p1, p2 = mk_prompts(cat)
        for kind, prompt in [("physical", p1), ("online", p2)]:
            pc, pl = backend_predict(prompt)
            rows.append({
                "TargetCategory": cat,
                "PromptType": kind,
                "Prompt": prompt,
                "PredCategory": pc,
                "PredLocation": pl,
                "CatMatch": (pc.lower() == cat.lower()),
                "LocIsOnline": (pl == "Online"),
                "LocIsGalle":  (pl == "Galle"),
            })
        if i % 20 == 0:
            print(f"Checked {i}/{len(ALL_CATS)} categories...")

df = pd.DataFrame(rows)
elapsed = time.time() - start
print(f"Done. {len(df)} rows in {elapsed:.1f}s.  (2 prompts per category)")

# save
OUT_DIR = os.path.join(ART_DIR, "eval")
os.makedirs(OUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUT_DIR, "category_smoke_test.csv")
df.to_csv(CSV_PATH, index=False, encoding="utf-8")
CSV_PATH


Done. 316 rows in 0.1s.  (2 prompts per category)


'artifacts_n\\eval\\category_smoke_test.csv'

### Loads model parts, makes 3 prompts per category, predicts category/location, grabs up to 5 same-category suggestions (if corpus exists), and saves results to category_smoke_test.csv.

In [ ]:


ART = Path("artifacts")  # adjust if different
featurizer = joblib.load(ART/"featurizer_cls.joblib")
cat_head   = joblib.load(ART/"cat_head.joblib")
loc_head   = joblib.load(ART/"loc_head.joblib")
encoders   = joblib.load(ART/"label_encoders.joblib")

# extract encoders no matter how they were saved
def _get_encs(obj):
    if isinstance(obj, dict):
        low = {str(k).lower(): v for k,v in obj.items()}
        return low.get("category") or low.get("cat"), low.get("location") or low.get("loc")
    if isinstance(obj, (list, tuple)) and len(obj)>=2:
        return obj[0], obj[1]
    if hasattr(obj, "encoders") and isinstance(obj.encoders, dict):
        low = {str(k).lower(): v for k,v in obj.encoders.items()}
        return low.get("category") or low.get("cat"), low.get("location") or low.get("loc")
    raise KeyError("Cannot extract encoders")
enc_cat, enc_loc = _get_encs(encoders)
all_categories = list(enc_cat.classes_)

# optional: load corpus/suggestions if you want to show neighbor suggestions
corpus = None
corpus_path = ART / "corpus.parquet"
if corpus_path.exists():
    try:
        corpus = pd.read_parquet(corpus_path)
    except Exception:
        try:
            corpus = pd.read_csv(str(corpus_path).replace(".parquet", ".csv"))
        except Exception:
            corpus = None

# helper: same boilerplate cleaner as backend (light)
import string
_PUNCT_TBL = str.maketrans("", "", string.punctuation)
def _strip_boilerplate(s: str) -> str:
    s0 = s.strip()
    s1 = re.sub(r"^start in\s+\w+(?:\s\w+)?\s*,\s*", "", s0, flags=re.I)
    s1 = re.sub(r",?\s*partner with local .*? bodies for credibility\.?", "", s1, flags=re.I)
    s1 = re.sub(r"\.?\s*apply for grants? (or )?accelerators? to scale\.?\s*$", "", s1, flags=re.I)
    s1 = re.sub(r"\s+", " ", s1).strip()
    return s1

def _diversity(lines):
    cores = [ _strip_boilerplate(x).lower().translate(_PUNCT_TBL) for x in lines if str(x).strip() ]
    return len(set(cores))

# simple predictor
def _predict(texts):
    Xv = featurizer.transform(pd.Series(texts, dtype=object))
    cats = enc_cat.inverse_transform(cat_head.predict(Xv))
    locs = enc_loc.inverse_transform(loc_head.predict(Xv))
    return [{"Category": c, "Location": l} for c,l in zip(cats,locs)]

# probes
district = "Colombo"
rows = []
for cat in all_categories:
    probes = [
        f"I want to start a {cat} business in {district}",
        f"I want to start a {cat} business online",
        f"I want to start a {cat} business",
    ]
    preds = _predict(probes)
    for prompt, pr in zip(probes, preds):
        # neighbor suggestions (if corpus exists): take top 5 within same category
        suggs = []
        if isinstance(corpus, pd.DataFrame) and len(corpus):
            # very simple: filter rows with same Category text, take first 20 suggestions
            cdf = corpus[ (corpus["Category"].astype(str).str.lower()==str(pr["Category"]).lower()) ]
            suggs = [str(x) for x in cdf["Suggestion"].dropna().head(20).tolist()]
        rows.append({
            "input": prompt,
            "predicted_category": pr["Category"],
            "predicted_location": pr["Location"],
            "suggestion_count": len(suggs[:5]),
            "suggestion_diversity_5": _diversity(suggs[:5]),
            "suggestions_5": " | ".join(suggs[:5]),
        })

df = pd.DataFrame(rows)
out = "category_smoke_test.csv"
df.to_csv(out, index=False)
print(f"Saved {len(df)} rows to {out}")


Saved 474 rows to category_smoke_test.csv
